In [ ]:
"""

Prepare the masks of the ice shelves based on BedMachine and other characteristics needed for basal melt parameterisations

"""

In [ ]:
import xarray as xr
import numpy as np
from pyproj import Transformer
import pandas as pd
from tqdm.notebook import trange, tqdm
import cc3d
import matplotlib as mpl


import multimelt.plume_functions as pf
import multimelt.box_functions as bf
import multimelt.useful_functions as uf
import multimelt.create_isf_mask_functions as isfmf

import distributed

In [ ]:
%matplotlib qt5

In [ ]:
map_lim = [-3000000,3000000]

#chunk_size = 700
chunk_size = False

READ IN DATA

In [ ]:
inputpath_data='/bettik/burgardc/DATA/SUMMER_PAPER/interim/'
inputpath_metadata='/bettik/burgardc/SCRIPTS/basal_melt_param/data/raw/MASK_METADATA/'
outputpath_mask ='/bettik/burgardc/DATA/SUMMER_PAPER/interim/ANTARCTICA_IS_MASKS/BedMachine_4km/'
outputpath_boxes = '/bettik/burgardc/DATA/SUMMER_PAPER/interim/BOXES/BedMachine_4km/'
outputpath_plumes = '/bettik/burgardc/DATA/SUMMER_PAPER/interim/PLUMES/BedMachine_4km/'

file_mask_orig = xr.open_dataset(inputpath_data+'BedMachine_v2_aggregated4km_allvars.nc')
file_mask_orig_cut = uf.cut_domain_stereo(file_mask_orig, map_lim, map_lim)

In [ ]:
file_msk = file_mask_orig_cut['mask_0_1_2']  #0 = ocean, 1 = ice shelves, 2 = grounded ice

In [ ]:
file_bed_orig = -1*file_mask_orig_cut['bed']
file_draft = (file_mask_orig_cut['thickness'] - file_mask_orig_cut['surface']).where(file_msk==1)
file_draft = file_draft.where(file_draft > 0, 0)
file_isf_conc = file_mask_orig_cut['isf_conc']
file_msk = file_mask_orig_cut['mask_0_1_2']  #0 = ocean, 1 = ice shelves, 2 = grounded ice

xx = file_msk['x']
yy = file_msk['y']

dx = abs(xx[1] - xx[0])
dy = abs(yy[1] - yy[0])

In [ ]:
IMBIE_file = xr.open_dataset(inputpath_data + 'Mask_Iceshelf_4km_IMBIE_withNisf.nc')

In [ ]:
IMBIE_file['Iceshelf_extrap'].where(IMBIE_file['Iceshelf_extrap'] == 2).plot()

In [ ]:
prelim_file = file_mask_basins['ID_IMBIE'].where(file_mask_basins['ID_IMBIE'] != 1, 159).where(file_msk == 1)
prelim_file = prelim_file.where(file_msk != 2, 0).where(file_msk != 0, 1)
prelim_file = prelim_file.where(prelim_file != 104, 103) # FRIS
prelim_file = prelim_file.where(prelim_file != 58, 57) # Ross

In [ ]:

whole_ds = isfmf.create_mask_and_metadata_isf(file_msk, -1*file_bed_orig, file_msk, -1*file_draft, file_isf_conc, False, 
                                          inputpath_data + 'Mask_Iceshelf_4km_IMBIE_withNisf.nc', outputpath_mask, 
                                          inputpath_metadata + 'iceshelves_metadata_Nico.txt', 
                                          inputpath_metadata+'GL_flux_rignot13.csv', mouginot_basins=True, variable_geometry=False,
                                          write_ismask = 'yes', write_groundmask = 'yes', write_outfile='yes',
                                          ground_point ='no',dist=40, add_fac=250, connectivity=4, threshold=4,
                                          write_metadata='yes')

# Write to netcdf
print('------- WRITE TO NETCDF -----------')
whole_ds.to_netcdf(outputpath_mask + 'BedMachinev2_4km_isf_masks_and_info_and_distance_oneFRIS.nc','w')


In [ ]:
whole_ds_tt = xr.open_dataset(outputpath_mask + 'BedMachinev2_4km_isf_masks_and_info_and_distance_oneFRIS.nc')


In [ ]:
file_bed_orig = -1*file_mask_orig_cut['bed']
file_draft = (file_mask_orig_cut['thickness'] - file_mask_orig_cut['surface']).where(file_msk==1)
file_draft = file_draft.where(file_draft > 0, 0)
file_isf_conc = file_mask_orig_cut['isf_conc']
file_msk = file_mask_orig_cut['mask_0_1_2']  #0 = ocean, 1 = ice shelves, 2 = grounded ice

xx = file_mask_orig_cut['x']
yy = file_mask_orig_cut['y']

whole_ds_tt = xr.open_dataset(outputpath_mask + 'BedMachinev2_4km_isf_masks_and_info_and_distance_oneFRIS.nc')

nonnan_Nisf = whole_ds_tt['Nisf'].where(np.isfinite(whole_ds_tt['front_bot_depth_max']), drop=True).astype(int)
file_isf_nonnan = whole_ds_tt.sel(Nisf=nonnan_Nisf)
rignot_isf = file_isf_nonnan.Nisf.where(np.isfinite(file_isf_nonnan['isf_area_rignot']), drop=True)
file_isf = file_isf_nonnan.sel(Nisf=rignot_isf)

In [ ]:
isf_var_of_int = whole_ds_tt[['ISF_mask', 'GL_mask', 'dGL', 'dIF', 'latitude', 'longitude', 'isf_name']]
out_2D, out_1D = bf.box_charac_file(file_isf['Nisf'],isf_var_of_int, -1*file_draft, file_isf_conc, outputpath_boxes, max_nb_box=10)
out_2D.to_netcdf(outputpath_boxes + 'BedMachine_4km_boxes_2D_oneFRIS.nc')
out_1D.to_netcdf(outputpath_boxes + 'BedMachine_4km_boxes_1D_oneFRIS.nc')
    


In [ ]:
plume_param_options = ['cavity','local','lazero'] #'cavity',, 'local','lazero'

plume_var_of_int = file_isf[['ISF_mask', 'GL_mask', 'IF_mask', 'dIF', 'dGL_dIF', 'latitude', 'longitude', 'front_ice_depth_avg']]

# Compute the ice draft
ice_draft_pos = file_draft

ice_draft_neg = -1*ice_draft_pos

plume_charac = pf.prepare_plume_charac(plume_param_options, ice_draft_pos, plume_var_of_int, grad_corr=5, dir_nb=16, extra_shift=2, dist_incl=1)
print('------ WRITE TO NETCDF -------')
plume_charac.to_netcdf(outputpath_plumes+'BedMachine_4km_plume_characteristics.nc')


In [ ]:
plume_param_options = ['new_lazero'] #'cavity',, 'local','lazero'

plume_var_of_int = file_isf[['ISF_mask', 'GL_mask', 'IF_mask', 'dIF', 'dGL_dIF', 'latitude', 'longitude', 'front_ice_depth_avg']]

# Compute the ice draft
ice_draft_pos = file_draft


ice_draft_neg = -1*ice_draft_pos

plume_charac_shift2 = pf.prepare_plume_charac(plume_param_options, ice_draft_pos, plume_var_of_int, grad_corr=5, dir_nb=16, extra_shift=2, dist_incl=1)
plume_charac_shift1 = pf.prepare_plume_charac(plume_param_options, ice_draft_pos, plume_var_of_int, grad_corr=5, dir_nb=16, extra_shift=1, dist_incl=1)
plume_charac = plume_charac_shift2.combine_first(plume_charac_shift1)
print('------ WRITE TO NETCDF -------')
#plume_charac.to_netcdf(outputpath_plumes+'BedMachine_4km_plume_characteristics.nc')
plume_charac.to_netcdf(outputpath_plumes+'BedMachine_4km_plume_characteristics_lazero_comparison_mixedshift.nc')
# smoothed was with grad_corr=2, dir_nb=16, extra_shift=2, dist_incl=1